# Stage 6 — h=1 Forecast Model & Leakage-Safe CV

Expanding walk-forward cross-validation with **all preprocessing models refit per fold**:

| Step | What | Fitted on |
|------|------|-----------|
| 1 | Intraday PCA basis | train rows only |
| 2 | Ridge macro baseline | train rows only |
| 3 | Pooled HMM | train rows only |
| 4 | Collapse to auction level | all (train+test) using train-fit models |
| 5 | Daily PCA for PC targets | train rows only |
| 6 | Random Forest | train rows only → predict test |

**Model:** Shallow `RandomForestRegressor` (max_depth=4, min_samples_leaf=5, n_estimators=300)  
appropriate for N≈220 auctions.

**Reads:**
- `data/cache/intraday_stage1.parquet`
- `data/cache/bloomberg.parquet`
- `data/cache/daily_curve.parquet`

Refs: Breiman (2001); López de Prado (2018) ch. 7.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import CACHE_DIR, TARGET, FEATURES, CV_N_SPLITS, CV_MIN_TRAIN
from src.model import walk_forward_leakage_safe, summarise_cv

intraday_df  = pd.read_parquet(CACHE_DIR / 'intraday_stage1.parquet')
bloomberg_df = pd.read_parquet(CACHE_DIR / 'bloomberg.parquet')
daily_df     = pd.read_parquet(CACHE_DIR / 'daily_curve.parquet')

print(f'Target: {TARGET}')
print(f'CV: {CV_N_SPLITS} folds, min_train={CV_MIN_TRAIN}')
print(f'Auctions: {bloomberg_df["auction_id"].nunique()}')

## 6a. Run leakage-safe walk-forward CV

This refits PCA + macro Ridge + HMM inside every fold. Expect ~1–3 min total.

In [ ]:
results = walk_forward_leakage_safe(
    intraday_df,
    bloomberg_df,
    daily_df,
    target=TARGET,
)

## 6b. Aggregate metrics

In [ ]:
mean_imp, metrics = summarise_cv(results, target=TARGET)
print('\nMetrics dict:', metrics)

## 6c. Diagnostics

In [ ]:
# Pooled actual vs predicted
actual  = np.concatenate([r['actual']  for r in results])
pred_rf = np.concatenate([r['pred_rf'] for r in results])
mask    = np.isfinite(actual)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(actual[mask], pred_rf[mask], alpha=0.4, s=15)
lim = max(abs(actual[mask]).max(), abs(pred_rf[mask]).max()) * 1.05
axes[0].plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'OOS: RF  R²={metrics["r2_rf"]:.3f}')

# Feature importance
mean_imp.head(15).sort_values().plot.barh(ax=axes[1])
axes[1].set_title('Mean feature importance (top 15)')

plt.tight_layout()
plt.show()

In [ ]:
# Per-fold R²
from sklearn.metrics import r2_score
fold_r2 = []
for i, r in enumerate(results):
    m = np.isfinite(r['actual'])
    r2 = r2_score(r['actual'][m], r['pred_rf'][m]) if m.sum() > 1 else np.nan
    fold_r2.append({'fold': i+1, 'n_train': r['fold_n_train'],
                    'n_test': r['fold_n_test'], 'r2_rf': r2})

fold_df = pd.DataFrame(fold_r2)
print(fold_df.to_string(index=False))

fold_df.set_index('fold')['r2_rf'].plot.bar(figsize=(8, 3))
plt.axhline(0, color='k', linewidth=0.8)
plt.title('OOS R² per fold')
plt.ylabel('R²')
plt.tight_layout()
plt.show()

## 6d. Persist OOS predictions

In [ ]:
oos_rows = []
for i, r in enumerate(results):
    df_t = r['df_test'].copy()
    df_t['pred_rf']   = r['pred_rf']
    df_t['pred_base'] = r['pred_base']
    df_t['fold']      = i + 1
    oos_rows.append(df_t)

oos_df = pd.concat(oos_rows, ignore_index=True)
oos_df.to_parquet(CACHE_DIR / 'oos_predictions.parquet', index=False)
print(f'Saved {len(oos_df)} OOS rows → cache/oos_predictions.parquet')